In [ ]:
import numpy as np
import pandas as pd
import glob
# from backend.models import PUBaggingClassifier
import pandas as pd
import geopandas as gpd
import numpy as np


In [ ]:
!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import pickle
from pulearn import BaggingPuClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import KNNImputer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score
from imblearn.over_sampling import SMOTE
import smote_variants as sv

# local imports
from backend.utils import csv2gdf


class PUBaggingClassifier:
    def __init__(self, 
                 training_df, 
                 training_variables,
                 target_variable,
                 weights=None,
                 correlation_threshold=None, 
                 n_estimators=200, 
                 max_samples=0.8, 
                 max_features=0.8, 
                 n_estimators_rf=200, 
                 max_depth_rf=12, 
                 min_samples_split_rf=10, 
                 min_samples_leaf_rf=5, 
                 max_features_rf='sqrt',
                 parallel=-1, 
                 random_state=22, 
                 scaled_probs=False,
                 use_smote=False,
                 smote_sampling_strategy=0.5,
                 use_smote_gan=False):
        
    # def __init__(self, training_df, training_variables, target_variable,weights=None, n_estimators=100, max_depth=8, 
    #              min_samples_split=10, min_samples_leaf=5, max_features='sqrt', parallel=-1, random_state=22, scaled_probs=False, criterion='gini'):
        

        """
        Initialize the PUBaggingClassifierModel with the given parameters.
        
        Parameters:
        - training_df (pd.DataFrame): The input DataFrame containing the data.
        - training_variables (list): The list of feature columns to be used.
        - target_variable (str): The target column for classification.
        - n_estimators (int): The number of trees in the bagging ensemble.
        - max_samples (float): The fraction of samples to use for each estimator.
        - max_features (float): The fraction of features to use for each estimator.
        """

        if isinstance(training_df, gpd.GeoDataFrame):
            gdf = training_df
        elif isinstance(training_df, pd.DataFrame):
            gdf=training_df
        else:
            gdf = gpd.read_file(training_df)
        
        self.df = pd.DataFrame(gdf).drop_duplicates().copy()
        
        self.correlation_threshold = correlation_threshold
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.max_features = max_features
        
        self.n_estimators_rf = n_estimators_rf
        self.max_depth_rf = max_depth_rf
        self.min_samples_split_rf = min_samples_split_rf
        self.min_samples_leaf_rf = min_samples_leaf_rf
        self.max_features_rf = max_features_rf
        
        
        self.random_state = random_state
        self.columns = training_variables
        self.target_variable = target_variable
        self.model = None
        self.n_jobs = parallel
        self.weights_cname = weights
        self.scaled_probs = scaled_probs
        self.use_smote=use_smote
        self.smote_sampling_strategy=smote_sampling_strategy
        self.use_smote_gan=use_smote_gan
        
        self.label_encoder = LabelEncoder()
        self.accuracy=None
        
        # Encode the target variable if it is categorical
        if self.df[self.target_variable].dtype == 'object' or self.df[self.target_variable].dtype.name == 'category':
            self.df[self.target_variable] = self.label_encoder.fit_transform(self.df[self.target_variable])
        elif self.df[self.target_variable].dtype.kind in {'i', 'u'}:
            unique_values = self.df[self.target_variable].unique()
            if len(unique_values) < 10:  # Arbitrary threshold for categorical-like integers
                self.df[self.target_variable] = self.label_encoder.fit_transform(self.df[self.target_variable])
            else:
                raise ValueError("You are using a classifier.Target variable must be categorical.")
        else:
            raise ValueError("You are using a classifier. Target variable must be categorical.")



    def apply_smote(self,X_train, y_train):
        print("Using SMOTE")
        smote = SMOTE(sampling_strategy=self.smote_sampling_strategy, random_state=42)  # 0.5 means minority class will be 50% of majority
        X_res, y_res = smote.fit_resample(X_train, y_train)
        return X_res, y_res
    
    def apply_smote_gan(self,X_train, y_train):
        print("Using SMOTE GAN")
        # Create SMOTE-GAN sampler
        smote_gan = sv.SMOTE_GAN()

        # Fit & resample
        X_res, y_res = smote_gan.sample(X_train.values, y_train.values)
        return X_res, y_res
    
    def fit(self):
        """
        Create and train a Bagging PU classifier model using the provided data.
        """
        try:

            if self.correlation_threshold is not None:
                # Step 1: Correlation filtering
                # Compute the correlation matrix for the features
                features_df = self.df[self.columns].copy()

                # Get the correlation matrix
                corr_matrix = features_df.corr()

                # Step 2: Identify features to drop based on correlation threshold
                to_drop = set()
                for i in range(len(corr_matrix.columns)):
                    for j in range(i):
                        if abs(corr_matrix.iloc[i, j]) > self.correlation_threshold:
                            colname = corr_matrix.columns[i]
                            to_drop.add(colname)

                # Step 3: Update the list of training variables by removing highly correlated features
                self.columns = [col for col in self.columns if col not in to_drop]

                print(f"Features after applying correlation threshold ({self.correlation_threshold}):")
                print(self.columns)

            if self.weights_cname in self.columns or self.weights_cname is None:
                features_nan = self.df[self.columns].copy()
            else:
                cols=self.columns.copy()
                cols.append(self.weights_cname)
                features_nan = self.df[cols].copy()
                

            # Identify categorical columns
            categorical_cols = features_nan.select_dtypes(include=['object', 'category']).columns

            # Apply One-Hot Encoding for categorical features
            if len(categorical_cols) > 0:
                features_nan = pd.get_dummies(features_nan, columns=categorical_cols)

            # Impute missing values
            imputer = KNNImputer(n_neighbors=4, weights="uniform")
            features = pd.DataFrame(imputer.fit_transform(features_nan), columns=features_nan.columns)
            # print(features.columns)
            print("Creating PUBaggingClassifierModel based on parameters:")
            for col in self.columns:
                print(col)

            # y = self.df[self.target_variable].values
            y=imputer.fit_transform(self.df[self.target_variable].values.reshape(-1, 1)).ravel()
            X_train=features.copy()
            y_train=y.copy()

            if self.use_smote:
                X_train,y_train=self.apply_smote(X_train,y_train)
            

            if self.use_smote_gan:
                X_train,y_train=self.apply_smote_gan(X_train,y_train)


            # pub = RandomForestClassifier(n_estimators=self.n_estimators_rf,
            #                                 max_depth=self.max_depth_rf,
            #                                 min_samples_split=self.min_samples_split_rf,
            #                                 min_samples_leaf=self.min_samples_leaf_rf,
            #                                 max_features=self.max_features_rf,
            #                                 random_state=self.random_state,
            #                                 class_weight="balanced",
            #                                 n_jobs=self.n_jobs)




                  # Define base XGBoost model
            pub = XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=self.random_state,
                scale_pos_weight=3,   # balances classes (similar to class_weight)
                n_jobs=self.n_jobs,
                use_label_encoder=False,
                eval_metric="logloss"
            )

            # # Define base LightGBM model
            # pub = LGBMClassifier(
            #     n_estimators=200,
            #     max_depth=-1,  # no limit
            #     learning_rate=0.05,
            #     num_leaves=31,
            #     subsample=0.8,
            #     colsample_bytree=0.8,
            #     random_state=self.random_state,
            #     class_weight="balanced",
            #     n_jobs=self.n_jobs)
            

            # Initialize BaggingPuClassifier
            self.model = BaggingPuClassifier(
                estimator=pub,
                n_estimators=self.n_estimators,
                max_samples=self.max_samples,
                max_features=self.max_features,
                random_state=self.random_state,
                n_jobs=self.n_jobs
            )

            weights = X_train[self.weights_cname].values.reshape(-1, 1) if self.weights_cname else None
            
            # Fit the model
            if weights is not None:
                self.model.fit(X_train[self.columns], y_train, sample_weight=weights.ravel())
            else:
                self.model.fit(X_train[self.columns], y_train)


            self.X_train=X_train
            self.y_train=y_train
 

            y_pred_full = self.model.predict(features[self.columns])
            y_pred_full = y_pred_full.astype(int) 


            # Add predicted target variable to the DataFrame
            if hasattr(self.label_encoder, 'classes_'):
                self.df[f'Predicted {self.target_variable}'] = self.label_encoder.inverse_transform(y_pred_full)
                self.df['Correct Prediction'] = self.df[self.target_variable] == self.df[f'Predicted {self.target_variable}']
                self.df[self.target_variable] = self.label_encoder.inverse_transform(self.df[self.target_variable])
            else:
                self.df[f'Predicted {self.target_variable}'] = y_pred_full
                self.df['Correct Prediction'] = self.df[self.target_variable] == y_pred_full

            probs = self.model.predict_proba(features[self.columns])

            # Handle probabilities scaling if required
            if self.scaled_probs:
                for i in range(probs.shape[1]):
                    mm_scaler = MinMaxScaler()
                    probs_scaled = mm_scaler.fit_transform(probs[:, i].reshape(-1, 1))
                    self.df[f"Scaled Probability(Label_{i})"] = probs_scaled
            else:
                for i in range(probs.shape[1]):
                    self.df[f"Probability(Label_{i})"] = probs[:, i]

        except Exception as e:
            raise ValueError(f"Error during model fitting: {str(e)}")
    
    def get_accuracy(self, test_df):
        
        X_test=test_df[self.columns].copy()
        y_test=test_df[self.target_variable].copy()
        
        imputer = KNNImputer(n_neighbors=4, weights="uniform")
        X_test = pd.DataFrame(imputer.fit_transform(X_test), columns=self.columns)
  
        y_pred = self.model.predict(X_test[self.columns])
        self.accuracy = accuracy_score(y_test, y_pred)
        print(f"Accuracy: {self.accuracy }")
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
        self.X_test=X_test
        self.y_test=y_test



    def grid_search_cv(self, param_grid, cv=3, scoring=None):
        # Prepare features and target
        features_nan = self.df[self.columns].copy()
        categorical_cols = features_nan.select_dtypes(include=['object', 'category']).columns
        if len(categorical_cols) > 0:
            features_nan = pd.get_dummies(features_nan, columns=categorical_cols)
        imputer = KNNImputer(n_neighbors=4, weights="uniform")
        features = pd.DataFrame(imputer.fit_transform(features_nan), columns=features_nan.columns)

        # Prepare target variable (already label encoded in __init__)
        y = self.df[self.target_variable].values

        # Define a base RandomForestClassifier
        # rf_pub = RandomForestClassifier(
        #     n_estimators=self.n_estimators_rf,
        #     max_depth=self.max_depth_rf,
        #     min_samples_split=self.min_samples_split_rf,
        #     min_samples_leaf=self.min_samples_leaf_rf,
        #     max_features=self.max_features_rf,
        #     random_state=self.random_state,
        #     class_weight="balanced",
        #     n_jobs=self.n_jobs
        # )


        # Define base XGBoost model
        pub = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=self.random_state,
            scale_pos_weight=3,   # balances classes (similar to class_weight)
            n_jobs=self.n_jobs,
            use_label_encoder=False,
            eval_metric="logloss"
        )

        # # Define base LightGBM model
        # pub = LGBMClassifier(
        #     n_estimators=200,
        #     max_depth=-1,  # no limit
        #     learning_rate=0.05,
        #     num_leaves=31,
        #     subsample=0.8,
        #     colsample_bytree=0.8,
        #     random_state=self.random_state,
        #     class_weight="balanced",
        #     n_jobs=self.n_jobs)
        

        # BaggingPuClassifier as main estimator
        base_model = BaggingPuClassifier(
            estimator=pub,
            n_estimators=self.n_estimators,
            max_samples=self.max_samples,
            max_features=self.max_features,
            random_state=self.random_state,
            n_jobs=self.n_jobs
        )

        # Choose scorer: default to accuracy
        metric = scoring if scoring else make_scorer(accuracy_score)

        # GridSearchCV
        grid = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            scoring=metric,
            cv=cv,
            verbose=1,
            n_jobs=self.n_jobs
        )
        grid.fit(features, y)
        self.model = grid.best_estimator_

        # Update best parameters in self if present in grid
        best_params = grid.best_params_
        self.n_estimators = best_params.get('n_estimators', self.n_estimators)
        self.max_samples = best_params.get('max_samples', self.max_samples)
        self.max_features = best_params.get('max_features', self.max_features)
        self.n_estimators_rf = best_params.get('estimator__n_estimators', self.n_estimators_rf)
        self.max_depth_rf = best_params.get('estimator__max_depth', self.max_depth_rf)
        self.min_samples_split_rf = best_params.get('estimator__min_samples_split', self.min_samples_split_rf)
        self.min_samples_leaf_rf = best_params.get('estimator__min_samples_leaf', self.min_samples_leaf_rf)
        self.max_features_rf = best_params.get('estimator__max_features', self.max_features_rf)
        self.random_state = best_params.get('random_state', self.random_state)

        print(f"Best parameters: {best_params}")
        print(f"Best score: {grid.best_score_}")
        return best_params, grid.best_score_

    def predict(self, df_loc):
        """
        Predict using the trained PUBagging model.
        
        Parameters:
        - df_loc (str): The location of the input file for prediction.
        
        Returns:
        - df (pd.DataFrame): DataFrame with predicted target values and probabilities.
        """
        try:
            # Load data and extract geometry
            _df = gpd.read_file(df_loc, engine= "fiona")
            _df["Longitude"] = _df['geometry'].x
            _df["Latitude"] = _df['geometry'].y
            df = pd.DataFrame(_df)

            # Prepare features
            features_nan = df[self.columns].copy()
            print(f"Using columns: {self.columns}")
            
            # Impute missing values
            imputer = KNNImputer(n_neighbors=4, weights="uniform")
            features = pd.DataFrame(imputer.fit_transform(features_nan), columns=self.columns)

            # Ensure features match training columns
            features = features.reindex(columns=self.columns)

            # Make predictions
            y_pred = self.model.predict(features)
            y_pred = np.round(y_pred).astype(int)  # Ensure integer output

            # Decode predictions if label encoder exists
            if hasattr(self.label_encoder, 'classes_'):
                df[f'Predicted_{self.target_variable}'] = self.label_encoder.inverse_transform(y_pred)
            else:
                df[f'Predicted_{self.target_variable}'] = y_pred

            # Get probabilities
            probs = self.model.predict_proba(features)
            
            # Add probabilities to DataFrame
            for i, class_label in enumerate(self.label_encoder.classes_ if hasattr(self.label_encoder, 'classes_') else range(probs.shape[1])):
                if self.scaled_probs:
                    df[f"Scaled_Probability({class_label})"] = MinMaxScaler().fit_transform(probs[:, i].reshape(-1, 1)).flatten()
                else:
                    df[f"Probability({class_label})"] = probs[:, i]

            print("Prediction completed successfully!")

        except Exception as e:
            print(f"Error during prediction: {str(e)}")
            raise ValueError(f"Error during prediction: Dataset is not trained or issue with the prediction file.")

        return df

    
    def save_model(self, loc):
        """
        Save the DataFrame, trained RandomForest model, and columns to the specified location.
        
        Parameters:
        - loc (str): Directory location to save the files.
        """
        try:
            os.makedirs(loc, exist_ok=True)
            unique_name = loc.split('/')[-1]
        
            # Save df to CSV
            self.df.to_csv(f'{loc}/{unique_name}_data.csv', index=False)
            
            # Save shp
            gdf = csv2gdf(self.df)
            gdf.to_file(f'{loc}/{unique_name}_data.gpkg', index=False)

            # Save model using pickle
            with open(f'{loc}/{unique_name}_model.pkl', 'wb') as f:
                pickle.dump(self.model, f)

            # Save columns to text file
            with open(f'{loc}/{unique_name}_columns.txt', 'w') as f:
                f.write('\n'.join(self.columns))

            with open(f'{loc}/{unique_name}_class.pkl', 'wb') as f:
                pickle.dump(self, f)
                
            print(f"Saved DataFrame, model, and columns to {loc}")
            
        except Exception as e:
            print(f"Error saving files: {str(e)}")
            raise ValueError(f"Error saving files: {str(e)}")



    @staticmethod
    def load_model(loc):
        """
        Load the saved PUBaggingClassifierModel instance from a pickle file.
        
        Parameters:
        - loc (str): Directory location where the class is saved.
        
        Returns:
        - PUBaggingClassifierModel: The loaded PUBaggingClassifierModel instance.
        """
        try:
            unique_name = loc.split('/')[-1]
            with open(f'{loc}/{unique_name}_class.pkl', 'rb') as f:
                loaded_instance = pickle.load(f)
            print("Class successfully loaded!")
            return loaded_instance
        except Exception as e:
            print(f"Error loading class: {str(e)}")
            return None


In [ ]:
training_df_filename=f"<DATA_ROOT>/CopperLithium/NorthAmerica/training_data/training_data_smotegan.csv"
training_df=pd.read_csv(training_df_filename)

In [ ]:
testing_df_filename=f"<DATA_ROOT>/CopperLithium/NorthAmerica/training_data/testing_data.csv"
testing_df=pd.read_csv(testing_df_filename)

In [ ]:


columns_to_train=[
       'trench_velocity_obliquity (degrees)', 
       'seafloor_age (Ma)',
       'distance_to_trench_edge (degrees)', 
       # 'trench_normal_angle (degrees)',
       'trench_velocity_orthogonal (cm/yr)',
       #   'subducting_plate_ID',
       'subducted_carbonates_volume (m)', 'trench_velocity (cm/yr)',
       'subducted_water_volume (m)',
       'subducting_plate_absolute_obliquity (degrees)',
       'convergence_obliquity (degrees)', 'co2_volume (m^3/m^2)',
       'subducted_sediment_volume (m)', 'convergence_rate (cm/yr)',
       'sediment_thickness (m)', 'carbonate_thickness (m)',
       # 'distance_from_trench_start (degrees)',
       'convergence_rate_orthogonal (cm/yr)',
       'subducting_plate_absolute_velocity_orthogonal (cm/yr)',
       'subducting_plate_absolute_velocity_parallel (cm/yr)',
       'subducted_plate_volume (m)', 'convergence_rate_parallel (cm/yr)',
       'subducting_plate_absolute_velocity (cm/yr)',
       # 'arc_segment_length (degrees)', 
       'seafloor_spreading_rate (km/Myr)',
       'trench_velocity_parallel (cm/yr)', 'distance_to_trench (km)',
       'water_thickness (m)', 'slab_flux (m^2/yr)',
       'crustal_thickness_mean (m)',
         'crustal_thickness_min (m)',
       'crustal_thickness_max (m)', 'crustal_thickness_median (m)',
       'crustal_thickness_std (m)',
       #   'crustal_thickness_n',
      #  'crustal_thickness_range (m)', 
       # 'magnetic_anomaly_mean (nT)',
       # 'magnetic_anomaly_min (nT)', 'magnetic_anomaly_max (nT)',
       # 'magnetic_anomaly_median (nT)', 'magnetic_anomaly_std (nT)',
       # 'magnetic_anomaly_n', 'magnetic_anomaly_range (nT)',
       'slab_dip (degrees)', 'arc_trench_distance (km)',
       'total_precipitation (km)', 
       'total_convergence (km)',
       # 'fz_distance', 'fz_magnitude', 'seamount_distance',
       #   'LIP_distance'
         ]




In [ ]:
model=PUBaggingClassifier(training_df=training_df,
                          training_variables=columns_to_train,
                          target_variable='label_binary',
                          weights='weights',
                          n_estimators=300
                        #   max_samples=1,
                        #   max_features=1,
                        #   n_estimators_rf=100,
                        #   max_depth_rf=8,
                        #   min_samples_split_rf=10,
                        #   min_samples_leaf_rf=5)
)

In [ ]:
model.fit()

In [ ]:
model.get_accuracy(testing_df)

In [ ]:
X_test=model.X_test.copy()
y_test=model.y_test

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

# Get predicted probabilities for positive class
y_scores = model.model.predict_proba(X_test)[:, 1]

# Compute precision, recall for all thresholds
precisions, recalls, thresholds = precision_recall_curve(model.y_test, y_scores)

# Average precision (area under PR curve)
ap = average_precision_score(model.y_test, y_scores)

# Plot
plt.figure(figsize=(7,5))
plt.plot(recalls, precisions, label=f"PR curve (AP={ap:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, precision_recall_curve

# Get predicted probabilities for the positive class (1.0)
y_scores = model.model.predict_proba(X_test)[:, 1]

# Example: use threshold = 0.3 instead of 0.5
threshold = 0.3
y_pred_adjusted = (y_scores >= threshold).astype(int)

print(f"Classification report at threshold={threshold}:")
print(classification_report(y_test, y_pred_adjusted))

# To visualize precision-recall tradeoff
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)

import matplotlib.pyplot as plt
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

# Get feature importances from all trees
all_importances = np.array([tree.feature_importances_ for tree in model.model.estimators_])
mean_importance = np.mean(all_importances, axis=0)

# Sort indices in descending order
sorted_idx = np.argsort(mean_importance)[::-1]

# Display sorted importances
for idx in sorted_idx:
    feat = columns_to_train[idx]
    imp = mean_importance[idx]
    print(f"{feat}: {imp:.4f}")


In [ ]:
data=gpd.read_file("<DATA_ROOT>/CopperLithium/NWMexico/PUBaggingModel/grid_data_fz_seamount_LIP.gpkg")

In [ ]:
data.columns

In [ ]:
 # Prepare features
features_nan =data[model.columns].copy()
print(f"Using columns: {model.columns}")


In [ ]:
features_nan

In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
# Ensure features match training columns
features = features_nan.reindex(columns=model.columns)

# Make predictions
y_pred = model.model.predict(features)
y_pred = np.round(y_pred).astype(int)  # Ensure integer output


# Get probabilities
probs = model.model.predict_proba(features)
# Decode predictions if label encoder exists
if hasattr(model.label_encoder, 'classes_'):
    features[f'Predicted_{model.target_variable}'] = model.label_encoder.inverse_transform(y_pred)
else:
    features[f'Predicted_{model.target_variable}'] = y_pred

# Add probabilities to DataFrame
for i, class_label in enumerate(model.label_encoder.classes_ if hasattr(model.label_encoder, 'classes_') else range(probs.shape[1])):
    if model.scaled_probs:
        features[f"Scaled_Probability({class_label})"] = MinMaxScaler().fit_transform(probs[:, i].reshape(-1, 1)).flatten()
    else:
        features[f"Probability({class_label})"] = probs[:, i]

In [ ]:
features['age (Ma)']=data['age (Ma)']
features['Latitude']=data['lat']
features['Longitude']=data['lon']

In [ ]:
features.to_csv(f"<DATA_ROOT>/CopperLithium/NorthAmerica/output/outputs.csv")